In [ ]:
import pandas as pd
import numpy as np
import diptest
from sklearn.mixture import GaussianMixture

# Load dataset
CSV_PATH = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_thickness_results.csv"
df = pd.read_csv(CSV_PATH)

print("=" * 70)
print(" MULTIMODALITY & GMM ANALYSIS PER PATIENT ")
print("=" * 70)

summary_rows = []

for patient_id, group in df.groupby('patient_id'):
    # Extract thickness values
    values = group['median_thickness_nm'].dropna().values
    
    if len(values) < 10:
        print(f"Patient {patient_id}: Too few samples ({len(values)}) for statistical testing.")
        continue

    # 1. Hartigan's Dip Test
    dip_stat, p_val = diptest.diptest(values)
    is_multimodal = p_val < 0.05

    # 2. Gaussian Mixture Model (Compare 1 vs 2 components using BIC)
    X = values.reshape(-1, 1)
    
    gmm1 = GaussianMixture(n_components=1, random_state=42).fit(X)
    gmm2 = GaussianMixture(n_components=2, random_state=42).fit(X)
    
    bic1 = gmm1.bic(X)
    bic2 = gmm2.bic(X)
    
    # Lower BIC indicates a better statistical fit
    best_fit = "Bimodal (2 Peaks)" if bic2 < bic1 else "Unimodal (1 Peak)"
    
    means_gmm2 = sorted(gmm2.means_.flatten())

    summary_rows.append({
        "Patient_ID": patient_id,
        "Sample_Count": len(values),
        "Dip_P_Value": round(p_val, 4),
        "Dip_Multimodal": is_multimodal,
        "GMM_Best_Fit": best_fit,
        "Peak_1_nm": round(means_gmm2[0], 1),
        "Peak_2_nm": round(means_gmm2[1], 1) if bic2 < bic1 else "N/A"
    })

    print(f"Patient {patient_id:<6} | Dip p-val: {p_val:.4f} ({'MULTIMODAL' if is_multimodal else 'UNIMODAL'}) | GMM Fit: {best_fit}")

# Convert to DataFrame and display
summary_df = pd.DataFrame(summary_rows)
print("=" * 70)
print("\nSummary Table:")
print(summary_df.to_string(index=False))

# Export summary table to CSV
OUTPUT_SUMMARY_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_multimodality_summary.csv"
summary_df.to_csv(OUTPUT_SUMMARY_CSV, index=False)
print(f"\nSaved statistical summary to '{OUTPUT_SUMMARY_CSV}'")